PILOT SKELETON - NOT RUN - synthetic data only - no result may be quoted from this notebook

# P2 pilot 02 - cue features on synthetic frames

Five small cue functions, one per cue family of the owner's design slides, applied to frames this notebook draws itself. numpy and OpenCV only, CPU only, fixed seed. It mounts no Drive and writes no file. Written by an agent through `build_notebooks.py`. Never executed. Not audited.

| Cue function | Design slide | Synthetic input |
|---|---|---|
| `contour_box_agreement` | 01 object appearance | sign-like polygon on a textured background |
| `shadow_band_edge_stat` | 02 shadows | dark slanted band |
| `saturated_region_stat` | 03 reflections | bright blob clipped to white |
| `global_local_luminance_residual` | 04 lighting | one gain on the whole frame |
| `flicker_stat` | 04 lighting and 05 video frames | horizontal stripes whose phase moves |

The functions return raw statistics. They set no threshold and take no decision. Not built here: viewpoint tracking, the reliability gates, the evidence accumulation of design slide 05, calibration, the explanation output and the abstain rule. The design slides state hypotheses, not a proven method. Synthetic frames can show only that the code runs and what each statistic responds to by construction.

In [ ]:
import platform
import sys
import time

import cv2
import numpy as np

SEED = 20261119   # one fixed seed, reused by every cell
H, W = 240, 320   # synthetic frame size in pixels, small on purpose
REPEATS = 30      # timed calls per function
TIMINGS = {}      # function name -> timing row
NAN = float("nan")
CHECK = "PILOT synthetic check, not a result"

cv2.setNumThreads(1)              # fixed OpenCV thread count
cv2.setRNGSeed(SEED)
if hasattr(cv2, "ocl"):
    cv2.ocl.setUseOpenCL(False)   # CPU code path only

print("PILOT SKELETON - synthetic frames only - nothing printed by this notebook is a result")
print("python", sys.version.split()[0], "| numpy", np.__version__, "| opencv", cv2.__version__)
print("platform", platform.platform(), "| seed", SEED, "| frame", W, "x", H)


def to_u8(values):
    """Round, clip to 0..255 and convert to uint8."""
    return np.clip(np.rint(values), 0, 255).astype(np.uint8)


def clip_box(box, shape):
    """Clip an (x, y, w, h) box to an image shape. Returns (x0, y0, x1, y1); empty when x1 <= x0 or y1 <= y0."""
    x, y, w, h = [int(v) for v in box]
    return max(0, x), max(0, y), min(int(shape[1]), x + w), min(int(shape[0]), y + h)


def time_call(name, fn, *args):
    """Call fn(*args) once to warm up, then REPEATS times with a timer; store and print one PILOT row.

    The row describes this runtime and one small synthetic input. It is not a latency result for
    a camera pipeline: decoding, the detector, tracking and queueing are all absent.
    """
    fn(*args)
    ms = []
    for _ in range(REPEATS):
        start = time.perf_counter()
        fn(*args)
        ms.append((time.perf_counter() - start) * 1000.0)
    TIMINGS[name] = dict(function=name, calls=REPEATS, seed=SEED, frame=f"{W}x{H}",
                         p50_ms=round(float(np.percentile(ms, 50)), 4),
                         p95_ms=round(float(np.percentile(ms, 95)), 4))
    print("PILOT timing, synthetic input, not a result:", TIMINGS[name])
    return TIMINGS[name]

## 2. Synthetic frames

The frames drawn below are stand-ins chosen so that each statistic has something to respond to. They are not road scenes, not attacks and not benign look-alikes.

In [ ]:
def make_background(rng):
    """Textured grey background: a smooth random field plus fine noise. Not a road scene."""
    coarse = rng.normal(0.0, 1.0, (H // 8 + 1, W // 8 + 1)).astype(np.float32)
    coarse = cv2.resize(coarse, (W, H), interpolation=cv2.INTER_CUBIC)
    fine = rng.normal(0.0, 1.0, (H, W)).astype(np.float32)
    return cv2.cvtColor(to_u8(120.0 + 20.0 * coarse + 6.0 * fine), cv2.COLOR_GRAY2BGR)


def sensor_noise(frame, rng, sigma=2.0):
    """Add independent Gaussian noise per pixel and channel (a crude stand-in for sensor noise)."""
    return to_u8(frame.astype(np.float32) + rng.normal(0.0, sigma, frame.shape).astype(np.float32))


def make_scene(rng, sides=8, radius=48):
    """Clean frame with one filled sign-like polygon near the centre.

    Returns (frame, box, mask): box = (x, y, w, h) of the drawn polygon, used as a stand-in for
    a detector box; mask = boolean polygon mask of the frame size.
    """
    frame = make_background(rng)
    cx = W // 2 + int(rng.integers(-20, 21))
    cy = H // 2 + int(rng.integers(-10, 11))
    angles = np.pi / sides + 2.0 * np.pi * np.arange(sides) / sides
    points = np.stack([cx + radius * np.cos(angles), cy + radius * np.sin(angles)], axis=1)
    polygon = np.rint(points).astype(np.int32).reshape(-1, 1, 2)
    cv2.fillPoly(frame, [polygon], (40, 40, 190))
    mask = np.zeros((H, W), np.uint8)
    cv2.fillPoly(mask, [polygon], 255)
    return sensor_noise(frame, rng), cv2.boundingRect(polygon), mask > 0


def add_shadow_band(frame, angle_deg=30.0, width=36.0, strength=0.5, soft_px=2.0):
    """Darken a slanted band through the frame centre (stand-in for a cast shadow band).

    Pixels in the band are multiplied by (1 - strength); the edge is a linear ramp soft_px wide.
    A real shadow also shifts colour and follows scene geometry; this one does neither.
    """
    yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
    theta = np.deg2rad(angle_deg)
    distance = np.abs((xx - W / 2.0) * np.cos(theta) + (yy - H / 2.0) * np.sin(theta))
    inside = np.clip((width / 2.0 - distance) / soft_px + 0.5, 0.0, 1.0)
    return to_u8(frame.astype(np.float32) * (1.0 - strength * inside)[:, :, None])


def add_specular_blob(frame, centre, sigma=7.0, peak=300.0):
    """Add a bright Gaussian blob that clips to white at its core (stand-in for a specular highlight)."""
    yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
    blob = np.exp(-((xx - centre[0]) ** 2 + (yy - centre[1]) ** 2) / (2.0 * sigma ** 2))
    return to_u8(frame.astype(np.float32) + peak * blob[:, :, None])


def apply_exposure_jump(frame, gain=1.3):
    """Multiply the whole frame by one gain (stand-in for an auto-exposure step). Clips at white."""
    return to_u8(frame.astype(np.float32) * gain)


def add_stripe_flicker(frame, phase, period_px=12.0, amplitude=14.0):
    """Add horizontal sinusoidal stripes at a phase given in cycles (stand-in for banding under a modulated light)."""
    rows = np.arange(H, dtype=np.float32)
    stripes = amplitude * np.sin(2.0 * np.pi * (rows / period_px + phase))
    return to_u8(frame.astype(np.float32) + stripes[:, None, None])


def make_sequence(rng, kind, n_frames=8):
    """Short frame list from one scene, with fresh noise per frame.

    kind: "static", "exposure" (one gain step halfway) or "stripes" (stripe phase moves each frame).
    Returns (frames, box, mask).
    """
    base, box, mask = make_scene(rng)
    frames = []
    for index in range(n_frames):
        frame = base
        if kind == "exposure" and index >= n_frames // 2:
            frame = apply_exposure_jump(frame)
        elif kind == "stripes":
            frame = add_stripe_flicker(frame, 0.37 * index)
        frames.append(sensor_noise(frame, rng))
    return frames, box, mask

## 3. Cue 1: contour-versus-box agreement (design slide 01)

In [ ]:
def contour_box_agreement(frame_bgr, box, pad=8):
    """Contour-versus-box agreement (design slide 01, object appearance).

    Measures: inside the box grown by pad pixels, an Otsu threshold on HSV saturation gives a
    foreground mask, and its largest external contour is compared with the box. Returned: IoU of
    the contour's bounding rectangle with the box, contour area over box area, contour area over
    convex-hull area, and the vertex count of a polygon approximation.
    Does not show: that a patch or sticker is present, or that the detector is wrong. Low
    agreement also follows from a grey or white sign face, low light, occlusion, clutter or a
    loose box. Viewpoint is not modelled and no threshold or decision is set here.
    """
    out = dict(found=False, box_iou=NAN, fill_ratio=NAN, solidity=NAN, vertices=0)
    x, y, w, h = [int(v) for v in box]
    x0, y0, x1, y1 = clip_box((x - pad, y - pad, w + 2 * pad, h + 2 * pad), frame_bgr.shape)
    if x1 <= x0 or y1 <= y0 or w <= 0 or h <= 0:
        return out
    hsv = cv2.cvtColor(frame_bgr[y0:y1, x0:x1], cv2.COLOR_BGR2HSV)
    saturation = cv2.GaussianBlur(np.ascontiguousarray(hsv[:, :, 1]), (5, 5), 0)
    _, foreground = cv2.threshold(saturation, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours = cv2.findContours(foreground, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    if len(contours) == 0:
        return out
    contour = max(contours, key=cv2.contourArea)
    area = float(cv2.contourArea(contour))
    cx, cy, cw, ch = cv2.boundingRect(contour)
    cx, cy = cx + x0, cy + y0   # back to frame coordinates
    inter_w = max(0, min(x + w, cx + cw) - max(x, cx))
    inter_h = max(0, min(y + h, cy + ch) - max(y, cy))
    inter = float(inter_w * inter_h)
    hull_area = float(cv2.contourArea(cv2.convexHull(contour)))
    polygon = cv2.approxPolyDP(contour, 0.02 * cv2.arcLength(contour, True), True)
    out.update(found=True, box_iou=inter / (w * h + cw * ch - inter), fill_ratio=area / (w * h),
               solidity=area / hull_area if hull_area > 0 else NAN, vertices=int(len(polygon)))
    return out

In [ ]:
# TIMING CELL 1 of 5 - contour_box_agreement - fixed seed
rng = np.random.default_rng(SEED)
clean, box, mask = make_scene(rng)
loose_box = (box[0] - 25, box[1], box[2] + 50, box[3])
print(CHECK, "| matching box:", contour_box_agreement(clean, box))
print(CHECK, "| loose box   :", contour_box_agreement(clean, loose_box))
time_call("contour_box_agreement", contour_box_agreement, clean, box)

## 4. Cue 2: shadow-band edge statistic (design slide 02)

In [ ]:
def shadow_band_edge_stat(frame_bgr, box, object_mask=None):
    """Shadow-band edge statistic (design slide 02, shadows).

    Measures, on the pixels of the box that lie in object_mask (the whole box if None): an Otsu
    split of luminance into a dark and a lit part; the dark fraction; the dark-to-lit luminance
    ratio; the shift in colour shares between the two parts (a cast shadow mainly scales
    intensity); and the mean luminance gradient on the dark/lit boundary over the luminance step
    (higher means a sharper edge).
    Does not show: that a shadow is present. Otsu splits any surface, also paint, dirt or plain
    noise on a uniform face, and the shape of the dark part (band or not) is not tested. It does
    not show what cast a shadow or whether it is physically plausible: one frame only, with no
    motion, sun or camera geometry.
    """
    out = dict(valid=False, dark_fraction=NAN, luminance_ratio=NAN, colour_shift=NAN, edge_sharpness=NAN)
    x0, y0, x1, y1 = clip_box(box, frame_bgr.shape)
    if x1 <= x0 or y1 <= y0:
        return out
    roi = frame_bgr[y0:y1, x0:x1]
    inside = np.ones(roi.shape[:2], bool) if object_mask is None else object_mask[y0:y1, x0:x1].astype(bool)
    if inside.shape != roi.shape[:2] or int(inside.sum()) < 16:
        return out
    luminance = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    threshold, _ = cv2.threshold(luminance[inside].reshape(-1, 1), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    dark = inside & (luminance <= threshold)
    lit = inside & (luminance > threshold)
    if not dark.any() or not lit.any():
        return out
    lum = luminance.astype(np.float32)
    mean_dark, mean_lit = float(lum[dark].mean()), float(lum[lit].mean())
    colour = roi.astype(np.float32)
    shares = colour / (colour.sum(axis=2, keepdims=True) + 1e-6)
    shift = float(np.abs(shares[dark].mean(axis=0) - shares[lit].mean(axis=0)).sum())
    gradient = cv2.magnitude(cv2.Sobel(lum, cv2.CV_32F, 1, 0, ksize=3, scale=0.25),
                             cv2.Sobel(lum, cv2.CV_32F, 0, 1, ksize=3, scale=0.25))
    boundary = (cv2.dilate(dark.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0) & lit
    step = mean_lit - mean_dark
    out.update(valid=True, dark_fraction=float(dark.sum()) / float(inside.sum()),
               luminance_ratio=mean_dark / max(mean_lit, 1e-6), colour_shift=shift,
               edge_sharpness=float(gradient[boundary].mean()) / step if boundary.any() and step > 0 else NAN)
    return out

In [ ]:
# TIMING CELL 2 of 5 - shadow_band_edge_stat - fixed seed
rng = np.random.default_rng(SEED)
clean, box, mask = make_scene(rng)
shadowed = add_shadow_band(clean)
print(CHECK, "| no band  :", shadow_band_edge_stat(clean, box, mask))
print(CHECK, "| with band:", shadow_band_edge_stat(shadowed, box, mask))
time_call("shadow_band_edge_stat", shadow_band_edge_stat, shadowed, box, mask)

## 5. Cue 3: saturated-region statistic (design slide 03)

In [ ]:
def saturated_region_stat(frame_bgr, box, v_min=250, s_max=40):
    """Saturated-region statistic (design slide 03, reflections).

    Measures, inside the box: the fraction of near-white pixels (HSV value >= v_min and
    saturation <= s_max), the area fraction of the largest connected near-white region, and the
    mean value contrast between that region and the rest of the box.
    Does not show: what caused the bright region. Sun glint, retroreflection, a wet surface, a
    lamp, white paint and a reflected-light attack can look alike in one RGB frame. Infrared
    light, which an IR-filtered camera may not record, is out of reach. No view, motion or
    exposure conditioning is done.
    """
    out = dict(valid=False, white_fraction=NAN, largest_region_fraction=NAN, value_contrast=NAN)
    x0, y0, x1, y1 = clip_box(box, frame_bgr.shape)
    if x1 <= x0 or y1 <= y0:
        return out
    hsv = cv2.cvtColor(frame_bgr[y0:y1, x0:x1], cv2.COLOR_BGR2HSV)
    value = hsv[:, :, 2]
    white = ((value >= v_min) & (hsv[:, :, 1] <= s_max)).astype(np.uint8)
    out.update(valid=True, white_fraction=float(white.mean()), largest_region_fraction=0.0)
    count, labels, stats, _ = cv2.connectedComponentsWithStats(white, connectivity=8)
    if count > 1:
        largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        region = labels == largest
        out["largest_region_fraction"] = float(stats[largest, cv2.CC_STAT_AREA]) / float(white.size)
        if not region.all():
            out["value_contrast"] = float(value[region].mean()) - float(value[~region].mean())
    return out

In [ ]:
# TIMING CELL 3 of 5 - saturated_region_stat - fixed seed
rng = np.random.default_rng(SEED)
clean, box, mask = make_scene(rng)
shiny = add_specular_blob(clean, (box[0] + box[2] // 2 - 12, box[1] + box[3] // 2 - 10))
print(CHECK, "| no blob  :", saturated_region_stat(clean, box))
print(CHECK, "| with blob:", saturated_region_stat(shiny, box))
time_call("saturated_region_stat", saturated_region_stat, shiny, box)

## 6. Cue 4: global-versus-local luminance residual (design slide 04)

In [ ]:
def global_local_luminance_residual(prev_bgr, curr_bgr, box):
    """Global-versus-local luminance residual (design slide 04, lighting).

    Measures: the log ratio of mean luminance between two frames inside the box (local term) and
    over the rest of the frame (global term), and their difference (residual). One gain on the
    whole frame moves both terms together; a light change confined to the box moves the local
    term only.
    Does not show: why the local term moved. Headlights, a passing shadow, object or camera
    motion, clipping at white and a projected pattern all leave a residual. The box is assumed
    to cover the same surface in both frames: no registration, tracking or camera response model.
    """
    out = dict(valid=False, global_log_ratio=NAN, local_log_ratio=NAN, residual=NAN)
    x0, y0, x1, y1 = clip_box(box, curr_bgr.shape)
    n_box = (x1 - x0) * (y1 - y0)
    n_all = curr_bgr.shape[0] * curr_bgr.shape[1]
    if x1 <= x0 or y1 <= y0 or prev_bgr.shape != curr_bgr.shape or n_box >= n_all:
        return out
    means = []
    for frame in (prev_bgr, curr_bgr):
        grey = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        box_sum = float(grey[y0:y1, x0:x1].sum(dtype=np.float64))
        rest_sum = float(grey.sum(dtype=np.float64)) - box_sum
        means.append((max(box_sum / n_box, 1e-6), max(rest_sum / (n_all - n_box), 1e-6)))
    local_term = float(np.log(means[1][0] / means[0][0]))
    global_term = float(np.log(means[1][1] / means[0][1]))
    out.update(valid=True, global_log_ratio=global_term, local_log_ratio=local_term,
               residual=local_term - global_term)
    return out

In [ ]:
# TIMING CELL 4 of 5 - global_local_luminance_residual - fixed seed
rng = np.random.default_rng(SEED)
clean, box, mask = make_scene(rng)
jump = apply_exposure_jump(clean)
local_light = add_specular_blob(clean, (box[0] + box[2] // 2, box[1] + box[3] // 2))
print(CHECK, "| gain on the whole frame:", global_local_luminance_residual(clean, jump, box))
print(CHECK, "| blob inside the box    :", global_local_luminance_residual(clean, local_light, box))
time_call("global_local_luminance_residual", global_local_luminance_residual, clean, jump, box)

## 7. Cue 5: frame-to-frame flicker statistic (design slides 04 and 05)

In [ ]:
def flicker_stat(frames_bgr, box):
    """Frame-to-frame flicker statistic (design slides 04 and 05, lighting and video frames).

    Measures, over a short list of frames and inside the box: the log of the row-mean luminance
    profile of each frame; its frame-to-frame change with the mean removed (so one gain on the
    whole frame largely cancels); the mean absolute size of that change; and the share of its
    non-DC spectral power that falls in the strongest row frequency (horizontal stripes
    concentrate the power, broadband noise spreads it).
    Does not show: that a light is modulated or that anything is malicious. Rolling shutter, LED
    signs, PWM lamps, compression and frame-rate aliasing can hide or imitate stripes. Frames are
    not registered and the box is fixed, so motion also changes the profile. The function reads
    the list it is given: a bounded causal buffer and a decision rule are not built.
    """
    out = dict(valid=False, frames=len(frames_bgr), mean_abs_row_change=NAN, stripe_peak_share=NAN)
    if len(frames_bgr) < 2:
        return out
    x0, y0, x1, y1 = clip_box(box, frames_bgr[0].shape)
    if x1 <= x0 or y1 - y0 < 8 or any(frame.shape != frames_bgr[0].shape for frame in frames_bgr):
        return out
    profiles = []
    for frame in frames_bgr:
        grey = cv2.cvtColor(frame[y0:y1, x0:x1], cv2.COLOR_BGR2GRAY)
        profiles.append(np.log(np.maximum(grey.mean(axis=1), 1e-3)))
    change = np.diff(np.stack(profiles), axis=0)
    change = change - change.mean(axis=1, keepdims=True)
    power = np.abs(np.fft.rfft(change, axis=1)[:, 1:]) ** 2
    share = power.max(axis=1) / np.maximum(power.sum(axis=1), 1e-12)
    out.update(valid=True, mean_abs_row_change=float(np.abs(change).mean()), stripe_peak_share=float(share.mean()))
    return out

In [ ]:
# TIMING CELL 5 of 5 - flicker_stat - fixed seed
for kind in ("static", "exposure", "stripes"):
    frames, box, mask = make_sequence(np.random.default_rng(SEED), kind)
    print(CHECK, "|", kind.ljust(8), ":", flicker_stat(frames, box))
time_call("flicker_stat", flicker_stat, frames, box)

## 8. What must replace the synthetic frames before any claim is made

Nothing printed by this notebook is evidence about attacks, detectors or road scenes. Before any claim is made:

1. **Real recordings** from datasets whose licence the owner confirmed (manifest in notebook 01), pinned by sha256, replace `make_scene` and `make_sequence`.
2. **A real detector**: boxes, classes and scores of a frozen detector replace the box of the drawn polygon, and tracked boxes replace the fixed box in the two-frame and multi-frame functions.
3. **Real attack and benign material**: physical or validly simulated attack clips for each cue family (patch or sticker, crafted shadow, reflected light, projected or flickering light, object-vanishing sequences) and benign look-alikes as hard negatives (tree and cloud shadows, glare, wet roads, retroreflection, auto-exposure jumps, lamp and sign flicker) replace the drawn band, blob, gain step and stripes.
4. **Held-out recordings**: splits by recording or route from the split manifest of notebook 01, never by frame, with thresholds and calibration fitted on recordings outside the test split.
5. **An internal analysis plan, self-hashed** before the first run on evaluation data: hypotheses, decisive metric, baselines, splits, stop rules.
6. **Baselines**: detector-only, each cue alone, and the published comparators named on the design slides once the literature catalog has confirmed them.
7. **Operating points**: recall at a matched false-alarm point and false alarms per hour of ordinary driving replace the raw statistics printed here.
8. **End-to-end cost**: p50 and p95 latency per frame including decoding, detector, tracking and queueing on the target hardware, and peak memory, replace the per-function timings.
9. **Sensors and conditions**: more than one camera, frame rate and compression setting; night, rain and low visibility; rolling shutter. Nothing tuned to this noise model and frame size carries over by default.
10. **Abstention counts**: the number of cases with no usable evidence (occlusion, failed segmentation, clipped pixels), reported next to every rate.
11. **A fresh-agent audit** that traces every number to a run log before anything is called a result.